## Étape 3 — Calcul des expositions et taux bruts de mortalité

Pour chaque âge entier x ∈ [20, 90] :
- **E_x** : exposition centrale (années-personnes)
- **D_x** : décès observés
- **q_x = 1 − exp(−D_x/E_x)** : taux brut annuel

In [ ]:
AGE_MIN, AGE_MAX = 20, 90
results = []
for age in range(AGE_MIN, AGE_MAX + 1):
    mask = (data['age_entree'] <= age) & (data['age_sortie'] > age)
    subset = data[mask]
    age_start = subset['date_naissance'] + pd.to_timedelta(age * 365, unit='D')
    age_end   = subset['date_naissance'] + pd.to_timedelta((age+1)*365, unit='D')
    obs_start = pd.concat([subset['date_entree'].reset_index(drop=True),
                           age_start.reset_index(drop=True)], axis=1).max(axis=1)
    obs_end   = pd.concat([subset['date_sortie'].reset_index(drop=True),
                           age_end.reset_index(drop=True)], axis=1).min(axis=1)
    exposition = ((obs_end - obs_start).dt.days / 365.25).clip(lower=0)
    E_x = exposition.sum()
    D_x = ((subset['cause_sortie']=='deces') & (subset['age_sortie'].astype(int)==age)).sum()
    mu_x = D_x / E_x if E_x > 0 else np.nan
    q_x  = 1 - np.exp(-mu_x) if not np.isnan(mu_x) else np.nan
    results.append({'age':age,'E_x':round(E_x,2),'D_x':int(D_x),'mu_x':mu_x,'q_x_brut':q_x})
table = pd.DataFrame(results)
total_E = table['E_x'].sum()
total_D = table['D_x'].sum()
n_vides  = table['E_x'].eq(0).sum()
n_faibles = (table['E_x'] < 10).sum()
print('Plage ages : {:}-{:}'.format(AGE_MIN, AGE_MAX))
print('Exposition totale  : {:,.0f} annees-personnes'.format(total_E))
print('Deces totaux       : {:,}'.format(int(total_D)))
print('Ages sans donnee   : {:}'.format(n_vides))
print('Ages E_x < 10      : {:}'.format(n_faibles))
print()
print(table[['age','E_x','D_x','q_x_brut']].head(15).to_string(index=False))
print('HELLO WORD')
